# 🔧 RAG Agent — Build & Evaluation

Builds a conversational RAG agent using **LangChain** with **multiple tools** and **memory**, then evaluates it with **LangSmith** LLM-as-judge.

## Pipeline
```
Part 1 — Agent Build
  1. Imports & config
  2. LangChain components (LLM, embeddings, Pinecone)
  3. Tools (search_transcripts, find_videos, get_video_info)
  4. Prompt + AgentExecutor
  5. Smoke test + multi-turn test
  6. Export config

Part 2 — LangSmith Evaluation
  7. predict_rag_answer wrapper
  8. Create evaluation dataset
  9. LLM-as-judge evaluators (accuracy, hallucination, relevance, helpfulness)
 10. Run evaluation + print summary
```

> **Requires:** `.env` with `OPENAI_API_KEY`, `PINECONE_API_KEY`, `LANGSMITH_API_KEY`


## 1. Install & Import Dependencies

In [1]:
# Run if needed
# !pip install langchain langchain-openai langchain-pinecone langgraph langsmith python-dotenv rank-bm25

In [1]:
import os
import json
from typing import TypedDict, List, Dict, Any, Annotated
from datetime import datetime

from dotenv import load_dotenv

# LangChain Core
from langchain.schema import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.tools import Tool, StructuredTool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory
from langchain.agents import create_openai_functions_agent, AgentExecutor
from pydantic import BaseModel, Field

# LangGraph (REQUIRED by Carlos for agents)
from langgraph.checkpoint.memory import MemorySaver
from langsmith import traceable
from langsmith import Client
from langsmith.evaluation import evaluate

# Pinecone
from pinecone import Pinecone

from langchain import hub

print("✅ All libraries imported")

import importlib.metadata
print("✅ All libraries imported")
print(f"   langgraph: {importlib.metadata.version('langgraph')}")


✅ All libraries imported
✅ All libraries imported
   langgraph: 0.2.16


## 2. Configuration

### 🔧 COST FIX #1: Force gpt-4o-mini usage

In [2]:
# Load environment variables
load_dotenv()

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

# LangSmith Configuration
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "youtube-rag-engineering-chatbot"
os.environ["LANGCHAIN_API_KEY"] = LANGSMITH_API_KEY or ""

# Pinecone Configuration
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "youtube-rag-mechanical-engineering")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "efficient-engineer-v3")

# Model Configuration - COST FIX: Force mini model
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL = os.getenv("CHAT_MODEL", "gpt-4o-mini")  # 🔧 HARDCODED to prevent accidental gpt-4o usage
JUDGE_MODEL     = os.getenv("JUDGE_MODEL", "gpt-4o-mini")   # LLM-as-judge model — swap to gpt-4o for higher accuracy

# Retrieval Configuration - COST FIX: Reduced from 5 to 3
TOP_K = 5  # 🔧 Fewer documents = fewer tokens

print("✅ Configuration loaded")
print(f"   Chat Model: {CHAT_MODEL} (COST OPTIMIZED)")
print(f"   Embedding Model: {EMBEDDING_MODEL}")
print(f"   Pinecone Index: {INDEX_NAME}")
print(f"   Namespace: {NAMESPACE}")
print(f"   LangSmith Tracing: Enabled")
print(f"   Advanced Retrieval: Top K={TOP_K} (COST OPTIMIZED)")

✅ Configuration loaded
   Chat Model: gpt-4o-mini (COST OPTIMIZED)
   Embedding Model: text-embedding-3-small
   Pinecone Index: youtube-rag-mechanical-engineering
   Namespace: efficient-engineer-v3
   LangSmith Tracing: Enabled
   Advanced Retrieval: Top K=5 (COST OPTIMIZED)


## 4. Initialize LangChain Components

In [3]:
# Initialize LLM with explicit mini model
llm = ChatOpenAI(
    model=CHAT_MODEL,
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)

# Initialize Embeddings
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)

# Initialize Pinecone VectorStore
vectorstore = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    namespace=NAMESPACE,
)

print("✅ LLM initialized")
print("✅ Embeddings initialized")
print("✅ VectorStore connected")

✅ LLM initialized
✅ Embeddings initialized
✅ VectorStore connected


## 5. Define Agent Tools (REQUIRED by Carlos)

Carlos requires "agents with **several tools**". Here we define 4 tools.

In [4]:
# Import StructuredTool for proper argument handling


# Define input schemas for tools
class SearchTranscriptsInput(BaseModel):
    query: str = Field(description="The search query to find relevant transcript content")

class GetVideoInfoInput(BaseModel):
    video_id: str = Field(description="The YouTube video ID to get information about")

class FindVideosInput(BaseModel):
    topic: str = Field(description="The topic to search for videos about")


def search_transcripts(query: str) -> str:
    """Search video transcripts for relevant content."""
    try:
        results = vectorstore.similarity_search(query, k=TOP_K)
        
        if not results:
            return "No relevant information found."
        
        formatted_results = []
        _last_retrieved_contexts = []  
        for i, doc in enumerate(results, 1):
            metadata = doc.metadata
            content = doc.page_content
            
            # Handle different metadata key formats
            video_id = metadata.get('video_id', metadata.get('videoId', 'unknown'))
            title = metadata.get('video_title', metadata.get('title', 'Unknown Video'))
            start_time = metadata.get('start_time', metadata.get('start', 0))
            
            formatted_results.append(
                f"[Result {i}]\n"
                f"Video: {title}\n"
                f"Time: {start_time}s\n"
                f"Content: {content}\n"
                f"Link: https://youtube.com/watch?v={video_id}&t={int(start_time)}s\n"
            )
            
      
        
        return "\n".join(formatted_results)
    
    except Exception as e:
        return f"Error searching transcripts: {str(e)}"


def get_video_info(video_id: str) -> str:
    """Get metadata about a specific video."""
    try:
        # Search for any chunk from this video
        results = vectorstore.similarity_search(
            query="",
            k=1,
            filter={"video_id": video_id}
        )
        
        if not results:
            return f"Video {video_id} not found in database."
        
        metadata = results[0].metadata
        
        info = (
            f"Title: {metadata.get('video_title', metadata.get('title', 'Unknown'))}\n"
            f"Channel: {metadata.get('channel', 'Unknown')}\n"
            f"Video ID: {video_id}\n"
            f"Link: https://youtube.com/watch?v={video_id}"
        )
        
        return info
    
    except Exception as e:
        return f"Error getting video info: {str(e)}"


def find_videos(topic: str) -> str:
    """Find videos related to a specific topic."""
    try:
        results = vectorstore.similarity_search(topic, k=TOP_K * 2)  # Get more to find unique videos
        
        if not results:
            return "No videos found on this topic."
        
        # Extract unique videos
        seen_videos = set()
        videos = []
        
        for doc in results:
            video_id = doc.metadata.get('video_id', doc.metadata.get('videoId'))
            if video_id and video_id not in seen_videos:
                seen_videos.add(video_id)
                videos.append({
                    'title': doc.metadata.get('video_title', doc.metadata.get('title', 'Unknown')),
                    'channel': doc.metadata.get('channel', 'Unknown'),
                    'video_id': video_id
                })
                
            if len(videos) >= TOP_K:  # Limit to TOP_K unique videos
                break
        
        formatted_videos = []
        for i, video in enumerate(videos, 1):
            formatted_videos.append(
                f"{i}. {video['title']}\n"
                f"   Channel: {video['channel']}\n"
                f"   Link: https://youtube.com/watch?v={video['video_id']}"
            )
        
        return "\n\n".join(formatted_videos)
    
    except Exception as e:
        return f"Error finding videos: {str(e)}"


# Create LangChain StructuredTools (fixes argument passing issues)
tools = [
    StructuredTool.from_function(
        func=search_transcripts,
        name="search_transcripts",
        description=(
            "Search YouTube video transcripts for information. Use this when the user asks "
            "questions about engineering concepts, definitions, or explanations. Returns "
            "relevant transcript segments with timestamps and video links."
        ),
        args_schema=SearchTranscriptsInput,
    ),
    StructuredTool.from_function(
        func=find_videos,
        name="find_videos",
        description=(
            "Find videos about a specific topic. Use this when the user asks 'which videos', "
            "'show me videos', or wants to browse content about a subject. Returns a list of "
            "relevant videos with titles and links."
        ),
        args_schema=FindVideosInput,
    ),
    StructuredTool.from_function(
        func=get_video_info,
        name="get_video_info",
        description=(
            "Get detailed metadata about a specific video by its ID. Use this when you have "
            "a video_id and need more information about that video. Returns title, channel, "
            "and link."
        ),
        args_schema=GetVideoInfoInput,
    ),
]

print(f"✅ Created {len(tools)} agent tools with StructuredTool:")
for tool in tools:
    print(f"   - {tool.name}: {tool.description[:60]}...")

✅ Created 3 agent tools with StructuredTool:
   - search_transcripts: Search YouTube video transcripts for information. Use this w...
   - find_videos: Find videos about a specific topic. Use this when the user a...
   - get_video_info: Get detailed metadata about a specific video by its ID. Use ...


## 6. Create Agent Prompt

In [5]:
system_message = """You are an expert mechanical engineering assistant with access to YouTube video transcripts.

Your role:
- Answer engineering questions using the transcript search tool
IMPORTANT: Base your answers ONLY on the information found in the transcript 
search results. Do not use outside knowledge. If the transcripts don't contain 
enough information, say so clearly.
- Provide accurate, detailed technical explanations
- Always cite sources with timestamps when available
- Suggest relevant videos when appropriate

Guidelines:
- Use search_transcripts for concept explanations and definitions
- Use find_videos when users ask "which videos" or "show me videos"
- Always include YouTube links with timestamps in your responses
- Be concise but thorough in explanations
- If information isn't found, say so clearly

Current date: {current_date}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

print("✅ Agent prompt template created")

✅ Agent prompt template created


## 7. Build Agent Executor


In [6]:
# LangGraph MemorySaver — provides in-process memory for multi-turn sessions
memory_saver = MemorySaver()

# Build the agent executor (needed for both chat and evaluation)
agent = create_openai_functions_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=5,
)
print("✅ agent_executor ready")

✅ agent_executor ready


## 8. Conversation Handler

In [7]:
def ask_agent(
    question: str,
    agent_executor: AgentExecutor,
    session_history: List[BaseMessage],
) -> str:
    """Ask the agent a question and return the response."""
    
    try:
        current_date = datetime.now().strftime("%Y-%m-%d")
        
        response = agent_executor.invoke({
            "input": question,
            "chat_history": session_history,
            "current_date": current_date,
        })
        
        answer = response.get("output", "")
        
        # Update history
        session_history.append(HumanMessage(content=question))
        session_history.append(AIMessage(content=answer))
        
        return answer
    
    except Exception as e:
        return f"Error: {str(e)}"


print("✅ Conversation handler defined")

✅ Conversation handler defined


## 9. Test Agent

In [8]:
# Single question smoke test
test_history  = []
test_question = "What is friction coefficient?"
print(f"\n❓ Question: {test_question}\n")

response = ask_agent(
    question        = test_question,
    agent_executor  = agent_executor,
    session_history = test_history,
)

print(f"\n💬 Response:\n{response}")



❓ Question: What is friction coefficient?


💬 Response:
The friction coefficient, often denoted as \( \mu \), is an empirical parameter that quantifies the frictional force between two contacting surfaces. It is determined experimentally and varies based on several factors, including the materials involved, surface roughness, cleanliness, temperature, and even heat treatment of the surfaces.

1. **Nature of Surfaces**: The coefficient of friction accounts for the characteristics of the two surfaces in contact. For example, clean, dry steel sliding on steel typically has a coefficient of kinetic friction around 0.5, but this value can drop significantly (to 0.1 or less) when lubricants are introduced, which create a thin film between the surfaces to reduce direct contact and friction (Understanding Friction, 306.56s, 496.72s).

2. **Experimental Measurement**: The coefficient of friction is often measured using devices like tribometers, which can employ setups such as a pin-on-disc con

## 10. Multi-turn conversation test

In [9]:
history  = []
questions = [
    'What is tensile strength?',
    'How does it relate to what you just explained?',
    'Which videos cover this topic?',
]

for q in questions:
    print(f'\n❓ {q}')
    print(f'💬 {ask_agent(q, agent_executor, history)[:400]}...')

print(f'\n🔢 History length: {len(history)} messages')


❓ What is tensile strength?
💬 Tensile strength is defined as the maximum stress that a material can withstand while being stretched or pulled before failing or breaking. It is a critical property in material science and engineering, particularly when evaluating materials for structural applications.

In the context of a tensile test, the ultimate tensile strength (UTS) is the highest point on the stress-strain curve, represent...

❓ How does it relate to what you just explained?
💬 Tensile strength and yield strength are closely related concepts in material science, particularly in the context of how materials respond to stress. 

1. **Tensile Strength**: This is the maximum stress a material can withstand while being stretched or pulled before it fails. It is represented by the ultimate tensile strength (UTS) on a stress-strain curve, which is the peak point of stress durin...

❓ Which videos cover this topic?
💬 Here are some videos that cover the topic of tensile strength and yield s

## 11. Export Agent for Deployment

In [10]:
import importlib.metadata

def get_version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return "not installed"

agent_config = {
    "created_at"         : datetime.now().isoformat(),
    "chat_model"         : CHAT_MODEL,
    "embedding_model"    : EMBEDDING_MODEL,
    "index_name"         : INDEX_NAME,
    "namespace"          : NAMESPACE,
    "top_k"              : TOP_K,
    "num_tools"          : len(tools),
    "tool_names"         : [tool.name for tool in tools],
    "agent_builder"      : "create_openai_functions_agent",
    "entrypoint_function": "ask_agent",
    "langsmith_project"  : os.environ.get("LANGCHAIN_PROJECT", ""),
    "package_versions"   : {
        "langchain"          : get_version("langchain"),
        "langchain-openai"   : get_version("langchain-openai"),
        "langchain-pinecone" : get_version("langchain-pinecone"),
        "langgraph"          : get_version("langgraph"),
        "pinecone"           : get_version("pinecone") or get_version("pinecone-client"),
    },
}

config_path = "../config/agent_config.json"
os.makedirs(os.path.dirname(config_path), exist_ok=True)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(agent_config, f, indent=2)

print(f"\n💾 Agent configuration saved to: {config_path}")
print(json.dumps(agent_config, indent=2))



💾 Agent configuration saved to: ../config/agent_config.json
{
  "created_at": "2026-03-12T16:14:18.946536",
  "chat_model": "gpt-4o-mini",
  "embedding_model": "text-embedding-3-small",
  "index_name": "youtube-rag-mechanical-engineering",
  "namespace": "efficient-engineer-v3",
  "top_k": 5,
  "num_tools": 3,
  "tool_names": [
    "search_transcripts",
    "find_videos",
    "get_video_info"
  ],
  "agent_builder": "create_openai_functions_agent",
  "entrypoint_function": "ask_agent",
  "langsmith_project": "youtube-rag-engineering-chatbot",
  "package_versions": {
    "langchain": "0.2.16",
    "langchain-openai": "0.1.23",
    "langchain-pinecone": "0.1.3",
    "langgraph": "0.2.16",
    "pinecone": "7.3.0"
  }
}


---

# 🧪 Part 2: RAG Agent Evaluation with LLM-as-Judge

The following sections evaluate the agent built above using LangSmith and an LLM-as-judge approach.

## 12. Prediction Function (wraps your agent for LangSmith)

> 💡 **What is `@traceable`?**  
> It tells LangSmith to record every call to this function as a traced run. You will see these runs in your LangSmith project dashboard, and the evaluation scores will be attached to them.

In [11]:
@traceable(name="rag_agent_predict")
def predict_rag_answer(example: dict) -> dict:
    """
    Wrapper that LangSmith's `evaluate()` will call for each dataset example.
    Returns both 'answer' and 'contexts' so all 4 evaluators can use it.
    """
    question = example["question"]
    
    # Retrieve contexts DIRECTLY here — don't rely on the global
    try:
        retrieved_docs = vectorstore.similarity_search(question, k=TOP_K)
        contexts = [doc.page_content for doc in retrieved_docs]
    except Exception:
        contexts = []

    try:
        response = agent_executor.invoke({
            "input": question,
            "chat_history": [],
            "current_date": datetime.now().strftime("%Y-%m-%d"),
        })
        answer = response.get("output", "")
    except Exception as e:
        answer = f"Error: {str(e)}"

    return {
        "answer": answer,
        "contexts": contexts,
    }


# Quick smoke test
test = predict_rag_answer({"question": "What is stress in engineering?"})
print("✅ Smoke test passed")
print(f"   Answer preview  : {test['answer'][:200]}...")
print(f"   Contexts found  : {len(test['contexts'])} chunks")

✅ Smoke test passed
   Answer preview  : In engineering, **stress** is defined as the internal resistance offered by a material to deformation when subjected to an external load. It is typically quantified as the force applied per unit area ...
   Contexts found  : 5 chunks


## 13. 📝 Create Evaluation Dataset in LangSmith

### ✏️ THIS IS WHAT YOU NEED TO EDIT

Add **5–10 question/answer pairs** that cover the topics in your video transcripts.

**Tips for writing good reference answers:**
- They don't need to be perfect word-for-word — the LLM judge scores semantic similarity
- Include key facts, definitions, or formulas the answer should contain
- Cover different question types: concept explanations, comparisons, "which videos" queries

> ⚠️ The dataset is created **once** in LangSmith. If you re-run this cell, it will try to create it again and fail. If that happens, just skip this cell — the dataset already exists.

In [30]:
# ────────────────────────────────────────────────────────────────────────────
# ✏️  EDIT THIS: Add your own Q&A pairs
# ────────────────────────────────────────────────────────────────────────────

DATASET_NAME = "mechanical-engineering-rag-eval-9"  # name shown in LangSmith

inputs = [
    "What is Finite element method?",
    "What is the difference between elastic deformation and plastic deformation?",
    "What is Poisson's ratio?",
    "What is tensile strength?",
    "What is compressive stress?",
    "What is strain energy?",
    "What is Turbulence in fluid dynamics?",
    "What is hardness in material science?",
    "What is buckling in columns?",
    "What is friction?"
    # ✏️  Add more questions here...
]

outputs = [
    # ✏️  Write the expected 'correct' answer for each question above
     "The Finite Element Method (FEM) is a numerical technique used to solve complex engineering problems by dividing the structure into smaller, simpler parts called elements. It allows for the analysis of stress, strain, and displacement in materials and structures.",
    "Elastic deformation is reversible deformation that disappears when the load is removed, while plastic deformation is permanent deformation that remains after the load is removed. Elastic deformation occurs below the yield strength, and plastic deformation occurs above it.",
    "Poisson's ratio (ν) is the ratio of lateral strain to axial strain when a material is stretched or compressed. It describes how much a material contracts sideways when stretched or expands sideways when compressed.",
    "Tensile strength is the maximum stress a material can withstand while being stretched before it breaks. It is determined from a tensile test and represents the highest point on the stress-strain curve.",
    "Compressive stress is the stress that occurs when a material is subjected to a force that pushes inward, reducing its length. It is calculated as σ = F/A, where the force acts to compress the material.",
    "Strain energy is the energy stored in a material when it is deformed under load. In the elastic region, this energy is recoverable when the load is removed, and it is equal to the area under the stress-strain curve up to that point.",
    "Turbulence is the chaotic, irregular motion of a fluid under high velocity or unstable flow conditions. It causes rapid changes in pressure and velocity and is characterized by swirling eddies and vortices. It typically occurs after laminar flow passes through a transition stage into fully turbulent flow.",
    "Hardness is the resistance of a material to localized plastic deformation, such as indentation or scratching. Common hardness tests include Brinell, Rockwell, and Vickers hardness tests.",
    "Buckling is a sudden sideways failure of a structural member subjected to compressive load. It often occurs in slender columns when the compressive force exceeds a critical value called the critical buckling load.",
    "Friction is the resistance to motion between two surfaces in contact. It acts parallel to the surfaces and opposes the relative motion of the objects."
    # ✏️  Add matching reference answers here...
]

assert len(inputs) == len(outputs), "Each question must have a matching reference answer!"

# ────────────────────────────────────────────────────────────────────────────

client = Client()

# Check if dataset already exists so we don't duplicate it
existing = list(client.list_datasets(dataset_name=DATASET_NAME))
if existing:
    print(f"ℹ️  Dataset '{DATASET_NAME}' already exists in LangSmith — skipping creation.")
    print(f"   Delete it in the LangSmith UI if you want to recreate it.")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Evaluation QA pairs for the Mechanical Engineering RAG Agent.",
    )
    client.create_examples(
        inputs=[{"question": q} for q in inputs],
        outputs=[{"answer": a} for a in outputs],
        dataset_id=dataset.id,
    )
    print(f"✅ Dataset '{DATASET_NAME}' created in LangSmith with {len(inputs)} examples.")
    print(f"   View it at: https://smith.langchain.com")

✅ Dataset 'mechanical-engineering-rag-eval-9' created in LangSmith with 10 examples.
   View it at: https://smith.langchain.com


## 14. Pull LLM-as-Judge Prompts from LangSmith Hub

These are standardised grader prompts maintained by LangChain. They tell the judge LLM exactly what to score and how.

| Prompt | Scores |
|--------|--------|
| `rag-answer-vs-reference` | Is the answer correct? (0 or 1) |
| `rag-answer-hallucination` | Is the answer grounded in retrieved docs? (0 or 1) |
| `rag-document-relevance` | Are the retrieved docs relevant? (0 or 1) |

In [31]:
grade_prompt_answer_accuracy  = hub.pull("langchain-ai/rag-answer-vs-reference")
grade_prompt_hallucinations   = hub.pull("langchain-ai/rag-answer-hallucination")
grade_prompt_doc_relevance    = hub.pull("langchain-ai/rag-document-relevance")

# One shared judge LLM instance
judge_llm = ChatOpenAI(model=JUDGE_MODEL, temperature=0)

print("✅ LLM-as-judge prompts loaded from LangSmith Hub")

✅ LLM-as-judge prompts loaded from LangSmith Hub


## 15. Define the 4 Evaluator Functions

> 💡 Each evaluator receives:
> - `run` → the output produced by your RAG agent (`run.outputs`)
> - `example` → the ground-truth row from your dataset (`example.inputs`, `example.outputs`)
>
> Each evaluator returns a `{"key": ..., "score": 0_or_1}` dict. LangSmith plots these scores.

In [32]:
# ── Evaluator 1: Answer Accuracy (vs. reference answer) ────────────────────

def answer_accuracy_evaluator(run, example) -> dict:
    """
    Metric: answer_accuracy
    Compares the agent's answer against the reference answer in the dataset.
    Score 1 = correct/similar, Score 0 = wrong/missing key info.
    Requires: reference answer in your dataset.
    """
    input_question = example.inputs["question"]
    reference      = example.outputs["answer"]
    prediction     = run.outputs["answer"]

    grader = grade_prompt_answer_accuracy | judge_llm
    result = grader.invoke({
        "question":       input_question,
        "correct_answer": reference,
        "student_answer": prediction,
    })
    score = result["Score"]
    print(f"  [answer_accuracy]    Q: {input_question[:60]:<60} → {score}")
    return {"key": "answer_accuracy", "score": score}


# ── Evaluator 2: Answer Hallucination ──────────────────────────────────────

def answer_hallucination_evaluator(run, example) -> dict:
    """
    Metric: answer_hallucination
    Checks whether the answer is grounded in the retrieved documents.
    Score 1 = grounded (not hallucinated), Score 0 = hallucinated.
    Does NOT need a reference answer.
    """
    input_question = example.inputs["question"]
    contexts       = run.outputs.get("contexts", [])
    prediction     = run.outputs["answer"]

    if not contexts:
        print(f"  [hallucination]      Q: {input_question[:60]:<60} → N/A (no contexts)")
        return {"key": "answer_hallucination", "score": None}  # can't evaluate without docs

    context_str = "\n\n".join(contexts)
    grader = grade_prompt_hallucinations | judge_llm
    result = grader.invoke({
        "documents":      context_str,
        "student_answer": prediction,
    })
    score = result["Score"]
    print(f"  [answer_hallucination] Q: {input_question[:60]:<60} → {score}")
    return {"key": "answer_hallucination", "score": score}


# ── Evaluator 3: Document Relevance ────────────────────────────────────────

def document_relevance_evaluator(run, example) -> dict:
    """
    Metric: document_relevance
    Checks whether the retrieved documents are relevant to the question.
    Score 1 = relevant, Score 0 = irrelevant/off-topic.
    Does NOT need a reference answer.
    """
    input_question = example.inputs["question"]
    contexts       = run.outputs.get("contexts", [])

    if not contexts:
        print(f"  [document_relevance] Q: {input_question[:60]:<60} → N/A (no contexts)")
        return {"key": "document_relevance", "score": None}

    context_str = "\n\n".join(contexts)
    grader = grade_prompt_doc_relevance | judge_llm
    result = grader.invoke({
        "question":  input_question,
        "documents": context_str,
    })
    score = result["Score"]
    print(f"  [document_relevance] Q: {input_question[:60]:<60} → {score}")
    return {"key": "document_relevance", "score": score}


# ── Evaluator 4: Answer Helpfulness (custom prompt — no hub needed) ─────────

HELPFULNESS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict but fair evaluator assessing whether an AI assistant's answer "
     "adequately addresses the user's question.\n"
     "Score 1 if the answer directly and helpfully addresses the question.\n"
     "Score 0 if the answer is off-topic, vague, or fails to address the question.\n"
     "Respond ONLY with a JSON object like: {{\"Score\": 1}} or {{\"Score\": 0}}"),
    ("human",
     "Question: {question}\n\nAnswer: {answer}\n\nScore:")
])

from langchain_core.output_parsers import JsonOutputParser

def answer_helpfulness_evaluator(run, example) -> dict:
    """
    Metric: answer_helpfulness
    Checks whether the answer actually addresses what the user asked.
    Score 1 = helpful, Score 0 = unhelpful/off-topic.
    Does NOT need a reference answer.
    """
    input_question = example.inputs["question"]
    prediction     = run.outputs["answer"]

    grader = HELPFULNESS_PROMPT | judge_llm | JsonOutputParser()
    try:
        result = grader.invoke({"question": input_question, "answer": prediction})
        score  = result.get("Score", 0)
    except Exception:
        score = 0

    print(f"  [answer_helpfulness] Q: {input_question[:60]:<60} → {score}")
    return {"key": "answer_helpfulness", "score": score}


print("✅ 4 evaluator functions defined")

✅ 4 evaluator functions defined


## 16. 🚀 Run the Full Evaluation

This runs all 4 evaluators across every example in your dataset and logs everything to LangSmith.

> 📌 After running, go to **https://smith.langchain.com** → **Projects** → **rag-agent-evaluation** to see your results.

In [33]:
print("🚀 Starting evaluation — this will take a few minutes...")
print(f"   Dataset    : {DATASET_NAME}")
print(f"   Judge LLM  : {JUDGE_MODEL}")
print(f"   LangSmith  : project = rag-agent-evaluation")
print()

experiment_results = evaluate(
    predict_rag_answer,
    data=DATASET_NAME,
    evaluators=[
        answer_accuracy_evaluator,          # Metric 1: correct vs. reference
        answer_hallucination_evaluator,     # Metric 2: grounded in retrieved docs
        document_relevance_evaluator,       # Metric 3: docs relevant to question
        answer_helpfulness_evaluator,       # Metric 4: answer addresses question
    ],
    experiment_prefix="rag-agent-eval",
    metadata={
        "agent_model":  CHAT_MODEL,
        "judge_model":  JUDGE_MODEL,
        "pinecone_index": INDEX_NAME,
        "top_k":        TOP_K,
        "description":  "Full evaluation of Mechanical Engineering RAG Agent with 4 LLM-as-judge metrics",
    },
)

print("\n✅ Evaluation complete!")
print("📊 View your results at: https://smith.langchain.com")

🚀 Starting evaluation — this will take a few minutes...
   Dataset    : mechanical-engineering-rag-eval-9
   Judge LLM  : gpt-4o-mini
   LangSmith  : project = rag-agent-evaluation

View the evaluation results for experiment: 'rag-agent-eval-469b7321' at:
https://smith.langchain.com/o/4838b4b3-1897-4213-9b1c-62965e576710/datasets/23b12232-ca09-4d9a-a670-1cf32bfcf0eb/compare?selectedSessions=d4007645-a77e-4ad5-b112-9e5643df3b82




0it [00:00, ?it/s]

  [answer_accuracy]    Q: What is Finite element method?                               → 1
  [answer_accuracy]    Q: What is the difference between elastic deformation and plast → 1
  [answer_accuracy]    Q: What is tensile strength?                                    → 1
  [answer_accuracy]    Q: What is Poisson's ratio?                                     → 1
  [answer_accuracy]    Q: What is hardness in material science?                        → 1
  [answer_accuracy]    Q: What is buckling in columns?                                 → 1
  [answer_accuracy]    Q: What is Turbulence in fluid dynamics?                        → 1
  [answer_accuracy]    Q: What is strain energy?                                       → 1
  [answer_accuracy]    Q: What is friction?                                            → 1
  [answer_accuracy]    Q: What is compressive stress?                                  → 1
  [answer_hallucination] Q: What is Turbulence in fluid dynamics?                        →

## 17. 📊 Print Local Summary of Results

In [34]:
# Collect scores per metric from the experiment results
metric_scores: Dict[str, List[float]] = {}

for result in experiment_results:
    # experiment_results is iterable; each item has .feedback_results
    for feedback in result.get("evaluation_results", {}).get("results", []):
        key   = feedback.key
        score = feedback.score
        if score is not None:
            metric_scores.setdefault(key, []).append(score)

print("=" * 55)
print("📊 EVALUATION SUMMARY")
print("=" * 55)

if metric_scores:
    for metric, scores in sorted(metric_scores.items()):
        avg   = sum(scores) / len(scores)
        count = len(scores)
        bar   = "█" * int(avg * 20) + "░" * (20 - int(avg * 20))
        print(f"  {metric:<28} {bar}  {avg:.0%}  ({count} examples)")
else:
    print("  (Run the cell above first to populate results)")

print("=" * 55)
print("\n📌 Full results with traces: https://smith.langchain.com")

📊 EVALUATION SUMMARY
  answer_accuracy              ████████████████████  100%  (10 examples)
  answer_hallucination         ████████████████████  100%  (10 examples)
  answer_helpfulness           ████████████████████  100%  (10 examples)
  document_relevance           ████████████████████  100%  (10 examples)

📌 Full results with traces: https://smith.langchain.com


## 18. 🔁 (Optional) Run Individual Metrics Separately

If you want to run only one metric at a time (e.g., to debug), use the cells below.

In [ ]:
# ── Run ONLY hallucination check ───────────────────────────────────────────
# Uncomment and run if you want to isolate this metric

# experiment_hallucination = evaluate(
#     predict_rag_answer,
#     data=DATASET_NAME,
#     evaluators=[answer_hallucination_evaluator],
#     experiment_prefix="rag-agent-hallucination-only",
# )

In [ ]:
# ── Run ONLY document relevance ────────────────────────────────────────────

# experiment_doc_relevance = evaluate(
#     predict_rag_answer,
#     data=DATASET_NAME,
#     evaluators=[document_relevance_evaluator],
#     experiment_prefix="rag-agent-doc-relevance-only",
# )

---

## 📚 Guide: How to Read Your LangSmith Dashboard

After running Section 8, go to **https://smith.langchain.com**:

1. **Projects** tab → click **rag-agent-evaluation**
2. Click the **Experiments** tab to see the run named `rag-agent-eval-...`
3. Click into the experiment to see a table with one row per question and columns for each metric score
4. Click any row to see the **full trace** — which tool was called, what docs were retrieved, and how the answer was generated

### Interpreting scores (all metrics are 0 or 1)

| Score | Meaning |
|-------|---------|
| `answer_accuracy = 1` | Agent's answer matches the reference answer ✅ |
| `answer_accuracy = 0` | Answer is wrong or missing key information ❌ |
| `answer_hallucination = 1` | Answer is grounded in retrieved docs ✅ |
| `answer_hallucination = 0` | Answer contains info NOT in the retrieved docs ❌ |
| `document_relevance = 1` | Retrieved chunks are relevant to the question ✅ |
| `document_relevance = 0` | Retrieval returned off-topic chunks ❌ |
| `answer_helpfulness = 1` | Answer addresses what was asked ✅ |
| `answer_helpfulness = 0` | Answer is vague, off-topic, or unhelpful ❌ |

### What to do when scores are low

| Low metric | Likely cause | How to fix |
|------------|-------------|------------|
| `document_relevance` | Poor vector search | Increase `TOP_K`, improve chunking, add metadata filters |
| `answer_hallucination` | LLM ignores retrieved docs | Strengthen system prompt: "only use information from the retrieved documents" |
| `answer_accuracy` | Wrong facts | Check if the answer is in your Pinecone index; review chunking |
| `answer_helpfulness` | Off-topic answers | Improve agent routing logic; add more specific tool descriptions |

---

## 🔄 How to Re-evaluate After Making Changes

1. Make your change (e.g., update TOP_K, change the system prompt, improve chunking)
2. Re-run cells 3, 6, and 8 — LangSmith will create a **new experiment** automatically
3. In LangSmith, use **Compare Experiments** to see if scores improved